# RailSafe — Kaggle Training & Evaluation

**Setup (2 things):**
1. **Add Data → GitHub** → `Akash-S-G/Railsafe` (code only, images gitignored)
2. **Add Input** → your datasets (different dir per dataset):
   - Railsense component data (`crossties/fasteners/fishplates/tracks`)
   - Multiclass data (`All_non-defective/Fastener_defective/Rail_defective`)
   - Surface faults (`Cracks/Flakings/Squats/...`) — e.g. `imenesabeur/test-xception`

**Trains 2 YOLO heads + full evaluation:**

| Head | Task | Classes | Data |
|---|---|---|---|
| `surface_classify` | `--task yolo` | 7 rail surface defects | 5153 imgs, video-grouped splits |
| `multiclass_cls` | `--task multiclass` | Normal/Fastener_defective/Rail_defective | 1185 imgs, date-grouped + stratified |

Evaluation (proper style per `docs/research/evaluation.md`): top1/top5 + **per-class accuracy table** + val/test + saved JSON.


In [ ]:
# 0. Environment check
import torch, os
from pathlib import Path
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('CWD:', os.getcwd())
INP = Path('/kaggle/input')
print('Kaggle Input datasets:')
for d in sorted(INP.iterdir()) if INP.exists() else []:
    print(' ', d.name)

In [ ]:
# 1. Copy datasets from /kaggle/input -> working dir (robust detection, dedup)
import shutil

def find_dir(markers, min_matches=1, name_filter=None):
    """Find input dir containing marker subdirs (any nesting)."""
    for top in sorted(INP.iterdir()):
        if not top.is_dir():
            continue
        if name_filter and name_filter(top.name.lower()):
            return top
        for sub in [top] + list(top.rglob('*')):
            try:
                if sub.is_dir():
                    hits = sum(1 for m in markers if (sub / m).exists())
                    if hits >= min_matches:
                        return sub if sub != top else top
            except (OSError, PermissionError):
                continue
    return None

def copy_tree(src, dst):
    dst.mkdir(parents=True, exist_ok=True)
    for p in src.rglob('*'):
        if p.is_file():
            out = dst / p.relative_to(src)
            out.parent.mkdir(parents=True, exist_ok=True)
            if not out.exists():
                shutil.copy2(p, out)

def count_imgs(d):
    return len(list(Path(d).rglob('*.jpg'))) + len(list(Path(d).rglob('*.JPEG')))

# Railsense: crossties/fasteners/fishplates/tracks
rs = find_dir(['crossties', 'fasteners'], min_matches=2, name_filter=lambda n: 'component' in n)
if rs:
    print(f'[input] railsense: {rs}')
    src = rs if (rs / 'crossties').exists() else (rs / 'data')
    if not (src / 'crossties').exists():
        src = next(d for d in rs.rglob('*') if d.is_dir() and (d / 'crossties').exists())
    for comp in ['crossties', 'fasteners', 'fishplates', 'tracks']:
        if (src / comp).exists():
            copy_tree(src / comp, Path('datasets/railsense') / comp)
            print(f'  {comp}: {count_imgs(Path("datasets/railsense")/comp)} imgs')
else:
    print('[input] railsense NOT found')

# Multiclass: All_non-defective/Fastener_defective/Rail_defective
mc = find_dir(['All_non-defective', 'Fastener_defective'], min_matches=2, name_filter=lambda n: 'multiclass' in n)
if mc:
    print(f'[input] multiclass: {mc}')
    src = mc if (mc / 'All_non-defective').exists() else next(d for d in mc.rglob('*') if d.is_dir() and (d / 'All_non-defective').exists())
    for cls in ['All_non-defective', 'Fastener_defective', 'Rail_defective']:
        if (src / cls).exists():
            copy_tree(src / cls, Path('datasets/kaggle_multiclass') / cls)
            print(f'  {cls}: {count_imgs(Path("datasets/kaggle_multiclass")/cls)} imgs')
else:
    print('[input] multiclass NOT found')

# Surface: Cracks/Flakings/Squats/Grooves/Joints/Shellings/Spallings
sf = find_dir(['Cracks', 'Flakings', 'Squats'], min_matches=3, name_filter=lambda n: 'surface' in n or 'xception' in n)
if sf:
    print(f'[input] surface: {sf}')
    src = sf if (sf / 'Cracks').exists() else next(d for d in sf.rglob('*') if d.is_dir() and (d / 'Cracks').exists())
    SFD = Path('datasets/track_surface_faults/Railway Track Surface Faults Dataset')
    for cls in ['Cracks', 'Flakings', 'Grooves', 'Joints', 'Shellings', 'Spallings', 'Squats']:
        if (src / cls).exists():
            copy_tree(src / cls, SFD / cls)
            print(f'  {cls}: {count_imgs(SFD/cls)} imgs')
else:
    print('[input] surface NOT found — add test-xception via Add Input')

In [ ]:
# 2. Install deps + build manifests + both YOLO layouts
!pip install -q ultralytics timm scikit-learn pyyaml 2>&1 | tail -n 2
!python ml/dataset_tools/convert_to_manifest.py --datasets railsense surface_faults kaggle_multiclass
!python ml/dataset_tools/prepare_yolo.py                  # 7-class surface
!python ml/dataset_tools/prepare_yolo.py --task multiclass # 3-class multiclass

In [ ]:
# 3. Dry-run verification (both heads)
!python scripts/train.py --task yolo --dry-run
!python scripts/train.py --task multiclass --dry-run

## 4. Train Head 1 — Surface (7-class rail surface defects)
`yolo11n-cls` (1.5M, 3.3 GFLOPs) — ~10 min on T4. Use `--model yolo11m.pt` for paper SOTA.

In [ ]:
!python scripts/train.py --task yolo --epochs 10 --model yolo11n.pt
!ls -lh runs/yolo/surface_classify*/weights/ | tail -n 3

## 5. Train Head 2 — Multiclass (3-class Normal/Fastener_defective/Rail_defective)
~2 min on T4 (1185 imgs). Complementary head: adds the missing Normal class.

In [ ]:
!python scripts/train.py --task multiclass --epochs 10 --model yolo11n.pt
!ls -lh runs/yolo/multiclass_cls*/weights/ | tail -n 3

## 6. Evaluation (proper style)
Per `docs/research/evaluation.md`: top1/top5 (val + test) + **per-class accuracy** + formatted tables + JSON saved.
Reports state split strategy, seed, threshold — printed below.

In [ ]:
import json, time
from pathlib import Path
from ultralytics import YOLO

RESULTS = Path('experiments/results')
RESULTS.mkdir(parents=True, exist_ok=True)

def per_class_accuracy(model, split_dir):
    """Per-class accuracy via batch prediction (proper style)."""
    out = {}
    for cls_dir in sorted(Path(split_dir).iterdir()):
        if not cls_dir.is_dir():
            continue
        imgs = list(cls_dir.glob('*.JPEG')) + list(cls_dir.glob('*.jpg'))
        if not imgs:
            continue
        rs = model.predict([str(p) for p in imgs], verbose=False)
        correct = sum(1 for r in rs if model.names[r.probs.top1] == cls_dir.name)
        out[cls_dir.name] = round(correct / len(imgs), 4)
    return out

def eval_head(weights, yolo_root, name):
    model = YOLO(str(weights))
    report = {'model': name, 'weights': str(weights),
              'split_strategy': 'grouped (video/date + stratified)', 'seed': 1337,
              'timestamp': time.strftime('%Y-%m-%d %H:%M')}
    for split in ['val', 'test']:
        m = model.val(data=str(yolo_root), split=split, verbose=False)
        report[split] = {'top1': round(float(m.top1), 4), 'top5': round(float(m.top5), 4)}
    report['per_class_test'] = per_class_accuracy(model, Path(yolo_root) / 'test')
    print(f"\n=== {name} ===")
    print('  split: grouped | seed 1337 | threshold: n/a (cls)')
    print(f"  val : top1 {report['val']['top1']:.4f}  top5 {report['val']['top5']:.4f}")
    print(f"  test: top1 {report['test']['top1']:.4f}  top5 {report['test']['top5']:.4f}")
    print('  per-class (test):')
    pc = report['per_class_test']
    for cls, acc in sorted(pc.items(), key=lambda x: -x[1]):
        bar = '█' * int(acc * 20)
        print(f'    {cls:<20} {acc:.4f} {bar}')
    macro = round(sum(pc.values()) / len(pc), 4) if pc else None
    print(f"    {'macro avg':<20} {macro}")
    report['macro_avg_test'] = macro
    (RESULTS / f'eval_{name}.json').write_text(json.dumps(report, indent=2))
    print(f'  saved -> {RESULTS}/eval_{name}.json')
    return report

surface_w = next(Path('runs/yolo').glob('surface_classify*/weights/best.pt'), None)
multiclass_w = next(Path('runs/yolo').glob('multiclass_cls*/weights/best.pt'), None)
rep_surface = eval_head(surface_w, 'datasets/yolo_surface', 'surface_classify') if surface_w else print('surface weights missing')
rep_mc = eval_head(multiclass_w, 'datasets/yolo_multiclass', 'multiclass_cls') if multiclass_w else print('multiclass weights missing')

## 7. Severity/Risk demo + Save models + Summary

In [ ]:
# Severity/risk engines (no training)
!python -c "from ml.severity.severity_engine import compute_severity; print('severity:', compute_severity(0.81,'Cracks',0.27,'rail'))"
!python -c "from ml.risk.risk_engine import compute_risk_v1; print('risk:', compute_risk_v1(0.68,0.85,0.27))"

# Save models to experiments/results/
import shutil
if surface_w:
    shutil.copy2(surface_w, RESULTS / 'best_surface_cls.pt'); print('saved', RESULTS / 'best_surface_cls.pt')
if multiclass_w:
    shutil.copy2(multiclass_w, RESULTS / 'best_multiclass_cls.pt'); print('saved', RESULTS / 'best_multiclass_cls.pt')

# Summary
print('\n=== ARTIFACTS ===')
for p in sorted(RESULTS.rglob('*')):
    if p.is_file():
        print(f'  {p} ({p.stat().st_size/1e6:.2f} MB)')
print('\nDownload: right sidebar Output -> Download, or:')
print('  cp experiments/results/best_*.pt /kaggle/working/')